# Model Selection and Ensemble Strategies

This notebook demonstrates advanced model selection techniques and ensemble strategies for the Neural-Forecast system.

## What you'll learn:
- Multi-criteria model selection
- Creating model ensembles
- Weighted averaging strategies
- Selecting models for different use cases
- Portfolio optimization for risk management

## 1. Setup and Load Sample CV Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import sys
sys.path.append('../..')

# Project imports
from cv.runner import summarize_cv
from uq.ensembles import create_ensemble, optimize_ensemble_weights

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

In [ ]:
# Create synthetic CV results for multiple models
np.random.seed(42)
n_samples = 1000
n_windows = 3

# Generate base truth
dates = pd.date_range('2024-01-01', periods=n_samples, freq='15min')
y_true = np.random.randn(n_samples) * 0.01

# Create CV results DataFrame
cv_results = pd.DataFrame({
    'unique_id': 'BTC',
    'ds': np.tile(dates, n_windows),
    'cutoff': np.repeat([dates[300], dates[400], dates[500]], n_samples),
    'y': np.tile(y_true, n_windows)
})

# Model configurations with different characteristics
models_config = {
    'NHITS_T': {'error': 0.002, 'bias': 0.0001, 'std_mult': 1.0, 'type': 'distributional'},
    'TiDE_T': {'error': 0.0022, 'bias': -0.0001, 'std_mult': 1.1, 'type': 'distributional'},
    'NBEATSx_T': {'error': 0.0025, 'bias': 0.0002, 'std_mult': 0.9, 'type': 'distributional'},
    'PatchTST_T': {'error': 0.0021, 'bias': 0.0, 'std_mult': 1.05, 'type': 'distributional'},
    'NHITS_MQ': {'error': 0.0023, 'bias': 0.0001, 'std_mult': 1.0, 'type': 'quantile'},
    'TiDE_IQ': {'error': 0.0024, 'bias': -0.0002, 'std_mult': 1.15, 'type': 'quantile'},
}

# Generate predictions for each model
for model_name, config in models_config.items():
    # Point predictions
    pred = np.tile(y_true, n_windows) + np.random.randn(len(cv_results)) * config['error'] + config['bias']
    cv_results[model_name] = pred
    
    # Prediction intervals
    for level in [80, 90, 95]:
        z = stats.norm.ppf(1 - (100-level)/200)
        std = config['error'] * config['std_mult'] * 1.5
        cv_results[f'{model_name}-lo-{level}'] = pred - z * std
        cv_results[f'{model_name}-hi-{level}'] = pred + z * std

print(f"Created CV results with {len(models_config)} models")
print(f"Shape: {cv_results.shape}")
print(f"Models: {list(models_config.keys())}")

## 2. Compute Individual Model Metrics

In [ ]:
# Compute metrics for all models
model_names = list(models_config.keys())
summary = summarize_cv(cv_results, model_names)

# Display leaderboard
print("=" * 70)
print("MODEL LEADERBOARD")
print("=" * 70)
display(summary['leaderboard'])

# Identify top performers
print("\nTop 3 models by sCRPS:")
top_3 = summary['leaderboard'].head(3)
for i, (model, row) in enumerate(top_3.iterrows(), 1):
    print(f"  {i}. {model}: sCRPS={row['sCRPS_mean']:.4f}")

## 3. Multi-Criteria Selection Framework

In [ ]:
def multi_criteria_selection(leaderboard, weights=None, constraints=None):
    """
    Select models based on multiple criteria with optional constraints.
    
    Args:
        leaderboard: DataFrame with model metrics
        weights: Dict of metric weights (sum to 1)
        constraints: Dict of metric constraints
    """
    if weights is None:
        weights = {
            'sCRPS': 0.4,
            'Coverage': 0.3,
            'Consistency': 0.2,
            'Bias': 0.1
        }
    
    scores = pd.DataFrame(index=leaderboard.index)
    
    # Normalize metrics to [0, 1] scale
    # sCRPS: lower is better
    scrps_norm = 1 - (leaderboard['sCRPS_mean'] - leaderboard['sCRPS_mean'].min()) / \
                 (leaderboard['sCRPS_mean'].max() - leaderboard['sCRPS_mean'].min())
    scores['sCRPS_score'] = scrps_norm
    
    # Coverage: deviation from nominal should be minimized
    coverage_dev = abs(leaderboard[['Coverage_80', 'Coverage_90', 'Coverage_95']].values - 
                      np.array([0.80, 0.90, 0.95])).mean(axis=1)
    coverage_norm = 1 - coverage_dev / coverage_dev.max()
    scores['Coverage_score'] = coverage_norm
    
    # Consistency: lower std is better
    consistency_norm = 1 - (leaderboard['sCRPS_std'] - leaderboard['sCRPS_std'].min()) / \
                      (leaderboard['sCRPS_std'].max() - leaderboard['sCRPS_std'].min())
    scores['Consistency_score'] = consistency_norm
    
    # Bias: closer to zero is better
    bias_norm = 1 - abs(leaderboard['Bias']) / abs(leaderboard['Bias']).max()
    scores['Bias_score'] = bias_norm
    
    # Compute weighted score
    scores['Total_score'] = (
        weights['sCRPS'] * scores['sCRPS_score'] +
        weights['Coverage'] * scores['Coverage_score'] +
        weights['Consistency'] * scores['Consistency_score'] +
        weights['Bias'] * scores['Bias_score']
    )
    
    # Apply constraints if provided
    if constraints:
        valid_models = leaderboard.index.tolist()
        if 'max_scrps' in constraints:
            valid_models = [m for m in valid_models 
                          if leaderboard.loc[m, 'sCRPS_mean'] <= constraints['max_scrps']]
        if 'min_coverage_80' in constraints:
            valid_models = [m for m in valid_models 
                          if leaderboard.loc[m, 'Coverage_80'] >= constraints['min_coverage_80']]
        scores = scores.loc[valid_models]
    
    return scores.sort_values('Total_score', ascending=False)

In [ ]:
# Apply multi-criteria selection
print("=" * 70)
print("MULTI-CRITERIA SELECTION")
print("=" * 70)

# Default weights
scores_default = multi_criteria_selection(summary['leaderboard'])
print("\nDefault weights (sCRPS=40%, Coverage=30%, Consistency=20%, Bias=10%):")
display(scores_default.round(3))

# Risk-averse weights (prioritize calibration)
risk_weights = {'sCRPS': 0.2, 'Coverage': 0.5, 'Consistency': 0.2, 'Bias': 0.1}
scores_risk = multi_criteria_selection(summary['leaderboard'], weights=risk_weights)
print("\nRisk-averse weights (Coverage=50%):")
display(scores_risk.head(3).round(3))

# Performance-focused weights
perf_weights = {'sCRPS': 0.7, 'Coverage': 0.1, 'Consistency': 0.15, 'Bias': 0.05}
scores_perf = multi_criteria_selection(summary['leaderboard'], weights=perf_weights)
print("\nPerformance-focused weights (sCRPS=70%):")
display(scores_perf.head(3).round(3))

## 4. Use Case Specific Selection

In [ ]:
print("=" * 70)
print("USE CASE SPECIFIC MODEL SELECTION")
print("=" * 70)

use_cases = {
    'High-Frequency Trading': {
        'description': 'Need fast, accurate point predictions',
        'weights': {'sCRPS': 0.3, 'Coverage': 0.1, 'Consistency': 0.3, 'Bias': 0.3},
        'constraints': {'max_scrps': 0.5}
    },
    'Risk Management': {
        'description': 'Need well-calibrated intervals for VaR',
        'weights': {'sCRPS': 0.2, 'Coverage': 0.6, 'Consistency': 0.15, 'Bias': 0.05},
        'constraints': {'min_coverage_80': 0.78}
    },
    'Portfolio Optimization': {
        'description': 'Need unbiased predictions with good uncertainty',
        'weights': {'sCRPS': 0.4, 'Coverage': 0.3, 'Consistency': 0.1, 'Bias': 0.2},
        'constraints': None
    },
    'Market Making': {
        'description': 'Need consistent, well-calibrated predictions',
        'weights': {'sCRPS': 0.3, 'Coverage': 0.3, 'Consistency': 0.3, 'Bias': 0.1},
        'constraints': {'max_scrps': 0.45}
    }
}

recommendations = {}
for use_case, config in use_cases.items():
    print(f"\n{use_case}:")
    print(f"  {config['description']}")
    
    scores = multi_criteria_selection(
        summary['leaderboard'],
        weights=config['weights'],
        constraints=config['constraints']
    )
    
    if len(scores) > 0:
        best = scores.index[0]
        recommendations[use_case] = best
        print(f"  Recommended: {best} (score: {scores.loc[best, 'Total_score']:.3f})")
    else:
        print(f"  No models meet constraints")

## 5. Simple Ensemble Strategies

In [ ]:
# Select top 3 models for ensemble
top_3_models = summary['leaderboard'].head(3).index.tolist()

print("=" * 70)
print("SIMPLE ENSEMBLE STRATEGIES")
print("=" * 70)
print(f"\nEnsemble members: {top_3_models}")

# Strategy 1: Equal weights
ensemble_equal = cv_results[top_3_models].mean(axis=1)
mae_equal = np.mean(np.abs(cv_results['y'] - ensemble_equal))

# Strategy 2: Inverse sCRPS weights
scrps_values = summary['leaderboard'].loc[top_3_models, 'sCRPS_mean'].values
weights_inv_scrps = (1 / scrps_values) / (1 / scrps_values).sum()
ensemble_weighted = (cv_results[top_3_models] * weights_inv_scrps).sum(axis=1)
mae_weighted = np.mean(np.abs(cv_results['y'] - ensemble_weighted))

# Strategy 3: Median (robust ensemble)
ensemble_median = cv_results[top_3_models].median(axis=1)
mae_median = np.mean(np.abs(cv_results['y'] - ensemble_median))

print("\nEnsemble Performance (MAE):")
print(f"  Equal weights:    {mae_equal:.6f}")
print(f"  Inverse sCRPS:    {mae_weighted:.6f}")
print(f"  Median (robust):  {mae_median:.6f}")

print("\nIndividual Model MAE:")
for model in top_3_models:
    mae_individual = np.mean(np.abs(cv_results['y'] - cv_results[model]))
    print(f"  {model}: {mae_individual:.6f}")

print("\nWeights for inverse sCRPS ensemble:")
for model, weight in zip(top_3_models, weights_inv_scrps):
    print(f"  {model}: {weight:.3f}")

## 6. Optimized Ensemble Weights

In [ ]:
from scipy.optimize import minimize

def optimize_ensemble_weights_crps(cv_results, model_names, y_col='y'):
    """
    Optimize ensemble weights to minimize CRPS.
    """
    n_models = len(model_names)
    
    # Extract predictions
    predictions = cv_results[model_names].values
    y_true = cv_results[y_col].values
    
    def objective(weights):
        # Ensure weights are positive and sum to 1
        weights = np.abs(weights) / np.abs(weights).sum()
        
        # Compute ensemble prediction
        ensemble_pred = (predictions * weights).sum(axis=1)
        
        # Compute MAE as proxy for CRPS (simplified)
        mae = np.mean(np.abs(y_true - ensemble_pred))
        return mae
    
    # Initial weights (equal)
    x0 = np.ones(n_models) / n_models
    
    # Constraints: weights sum to 1
    constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
    
    # Bounds: weights between 0 and 1
    bounds = [(0, 1) for _ in range(n_models)]
    
    # Optimize
    result = minimize(objective, x0, method='SLSQP', 
                     bounds=bounds, constraints=constraints)
    
    return result.x / result.x.sum()  # Normalize to ensure sum to 1

# Optimize weights
optimal_weights = optimize_ensemble_weights_crps(cv_results, top_3_models)

print("=" * 70)
print("OPTIMIZED ENSEMBLE WEIGHTS")
print("=" * 70)
print("\nOptimal weights (minimize MAE):")
for model, weight in zip(top_3_models, optimal_weights):
    print(f"  {model}: {weight:.3f}")

# Compare performance
ensemble_optimal = (cv_results[top_3_models] * optimal_weights).sum(axis=1)
mae_optimal = np.mean(np.abs(cv_results['y'] - ensemble_optimal))

print(f"\nOptimized ensemble MAE: {mae_optimal:.6f}")
print(f"Improvement over equal weights: {(mae_equal - mae_optimal)/mae_equal*100:.1f}%")

## 7. Model Diversity Analysis

In [ ]:
# Analyze model diversity for ensemble effectiveness
print("=" * 70)
print("MODEL DIVERSITY ANALYSIS")
print("=" * 70)

# Compute correlation matrix
model_predictions = cv_results[model_names]
correlation_matrix = model_predictions.corr()

# Visualize correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap
ax = axes[0]
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', 
            cmap='coolwarm', center=0.5, vmin=0, vmax=1,
            square=True, ax=ax)
ax.set_title('Model Prediction Correlations')

# Diversity scores
ax = axes[1]
diversity_scores = {}
for model in model_names:
    # Average correlation with other models
    other_models = [m for m in model_names if m != model]
    avg_corr = correlation_matrix.loc[model, other_models].mean()
    diversity_scores[model] = 1 - avg_corr  # Higher score = more diverse

models = list(diversity_scores.keys())
scores = list(diversity_scores.values())
colors = ['green' if s > 0.3 else 'orange' if s > 0.2 else 'red' for s in scores]

bars = ax.bar(range(len(models)), scores, color=colors, alpha=0.7)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=45)
ax.set_ylabel('Diversity Score')
ax.set_title('Model Diversity Scores\n(Higher = More Unique)')
ax.axhline(y=0.3, color='green', linestyle='--', alpha=0.5, label='Good diversity')
ax.axhline(y=0.2, color='orange', linestyle='--', alpha=0.5, label='Moderate diversity')
ax.legend()

plt.tight_layout()
plt.show()

print("\nDiversity Analysis:")
print("  High diversity (>0.3): Good for ensemble")
print("  Moderate diversity (0.2-0.3): Acceptable for ensemble")
print("  Low diversity (<0.2): Limited ensemble benefit")

# Recommend ensemble composition
high_diversity = [m for m, s in diversity_scores.items() if s > 0.25]
if len(high_diversity) >= 3:
    print(f"\nRecommended diverse ensemble: {high_diversity[:3]}")
else:
    print(f"\nWarning: Low model diversity. Consider training models with different architectures.")

## 8. Distributional vs Quantile Model Selection

In [ ]:
# Separate models by type
dist_models = [m for m, cfg in models_config.items() if cfg['type'] == 'distributional']
quant_models = [m for m, cfg in models_config.items() if cfg['type'] == 'quantile']

print("=" * 70)
print("DISTRIBUTIONAL vs QUANTILE MODELS")
print("=" * 70)

# Compare performance
dist_metrics = summary['leaderboard'].loc[dist_models]
quant_metrics = summary['leaderboard'].loc[quant_models]

print("\nDistributional Models (StudentT):")
print(f"  Models: {dist_models}")
print(f"  Avg sCRPS: {dist_metrics['sCRPS_mean'].mean():.4f}")
print(f"  Avg Coverage 90%: {dist_metrics['Coverage_90'].mean():.3f}")
print(f"  Best: {dist_metrics['sCRPS_mean'].idxmin()}")

print("\nQuantile Models (MQLoss/IQLoss):")
print(f"  Models: {quant_models}")
print(f"  Avg sCRPS: {quant_metrics['sCRPS_mean'].mean():.4f}")
print(f"  Avg Coverage 90%: {quant_metrics['Coverage_90'].mean():.3f}")
print(f"  Best: {quant_metrics['sCRPS_mean'].idxmin()}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# sCRPS comparison
ax = axes[0]
x = np.arange(len(dist_models))
width = 0.35
ax.bar(x - width/2, dist_metrics['sCRPS_mean'].values, width, 
       label='Distributional', color='blue', alpha=0.7)
ax.bar(x + width/2, quant_metrics['sCRPS_mean'].values[:len(dist_models)], width, 
       label='Quantile', color='green', alpha=0.7)
ax.set_xlabel('Model Index')
ax.set_ylabel('sCRPS')
ax.set_title('sCRPS: Distributional vs Quantile')
ax.legend()

# Coverage comparison
ax = axes[1]
coverage_levels = ['Coverage_80', 'Coverage_90', 'Coverage_95']
dist_cov = dist_metrics[coverage_levels].mean().values
quant_cov = quant_metrics[coverage_levels].mean().values
x = np.arange(len(coverage_levels))
ax.bar(x - width/2, dist_cov, width, label='Distributional', color='blue', alpha=0.7)
ax.bar(x + width/2, quant_cov, width, label='Quantile', color='green', alpha=0.7)
ax.axhline(y=0.80, xmin=0, xmax=0.3, color='black', linestyle='--', alpha=0.5)
ax.axhline(y=0.90, xmin=0.35, xmax=0.65, color='black', linestyle='--', alpha=0.5)
ax.axhline(y=0.95, xmin=0.7, xmax=1.0, color='black', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(['80%', '90%', '95%'])
ax.set_ylabel('Empirical Coverage')
ax.set_title('Coverage: Distributional vs Quantile')
ax.legend()

plt.tight_layout()
plt.show()

print("\nRecommendations:")
print("  - Use distributional models when you need:")
print("    • Full probability distributions")
print("    • PIT analysis capability")
print("    • Consistent uncertainty across all quantiles")
print("  - Use quantile models when you need:")
print("    • Specific percentile predictions")
print("    • Robustness to distribution misspecification")
print("    • Asymmetric risk management")

## 9. Hierarchical Model Selection

In [ ]:
def hierarchical_selection(leaderboard, cv_results, model_names):
    """
    Hierarchical selection: best overall → best per category → ensemble.
    """
    selection = {}
    
    # Level 1: Best overall
    selection['best_overall'] = leaderboard.index[0]
    
    # Level 2: Best by type
    dist_models = [m for m in model_names if '_T' in m]
    quant_models = [m for m in model_names if '_MQ' in m or '_IQ' in m]
    
    if dist_models:
        dist_subset = leaderboard.loc[dist_models]
        selection['best_distributional'] = dist_subset['sCRPS_mean'].idxmin()
    
    if quant_models:
        quant_subset = leaderboard.loc[quant_models]
        selection['best_quantile'] = quant_subset['sCRPS_mean'].idxmin()
    
    # Level 3: Best diverse pair
    correlations = cv_results[model_names].corr()
    min_corr = 1.0
    best_pair = None
    
    for i, m1 in enumerate(model_names):
        for m2 in model_names[i+1:]:
            corr = correlations.loc[m1, m2]
            # Weight by performance
            perf_penalty = (leaderboard.loc[m1, 'sCRPS_mean'] + 
                          leaderboard.loc[m2, 'sCRPS_mean']) / 2
            score = corr * (1 + perf_penalty)
            
            if score < min_corr:
                min_corr = score
                best_pair = (m1, m2)
    
    selection['best_diverse_pair'] = best_pair
    
    # Level 4: Optimal ensemble size
    ensemble_performance = []
    for n in range(2, min(6, len(model_names) + 1)):
        top_n = leaderboard.head(n).index.tolist()
        ensemble_pred = cv_results[top_n].mean(axis=1)
        mae = np.mean(np.abs(cv_results['y'] - ensemble_pred))
        ensemble_performance.append((n, mae, top_n))
    
    best_ensemble = min(ensemble_performance, key=lambda x: x[1])
    selection['optimal_ensemble_size'] = best_ensemble[0]
    selection['optimal_ensemble'] = best_ensemble[2]
    
    return selection

# Apply hierarchical selection
hierarchy = hierarchical_selection(summary['leaderboard'], cv_results, model_names)

print("=" * 70)
print("HIERARCHICAL MODEL SELECTION")
print("=" * 70)

print("\nSelection Hierarchy:")
print(f"\n1. Best Overall Model:")
print(f"   {hierarchy['best_overall']}")

print(f"\n2. Best by Type:")
if 'best_distributional' in hierarchy:
    print(f"   Distributional: {hierarchy['best_distributional']}")
if 'best_quantile' in hierarchy:
    print(f"   Quantile: {hierarchy['best_quantile']}")

print(f"\n3. Best Diverse Pair:")
print(f"   {hierarchy['best_diverse_pair']}")

print(f"\n4. Optimal Ensemble:")
print(f"   Size: {hierarchy['optimal_ensemble_size']} models")
print(f"   Members: {hierarchy['optimal_ensemble']}")

## 10. Final Recommendations

In [ ]:
print("=" * 70)
print("FINAL MODEL SELECTION RECOMMENDATIONS")
print("=" * 70)

# Create recommendation matrix
recommendations_matrix = pd.DataFrame({
    'Scenario': [
        'Production (Conservative)',
        'Production (Balanced)',
        'Production (Aggressive)',
        'Research/Development',
        'Backtesting'
    ],
    'Strategy': [
        f"Ensemble of top 3: {hierarchy['optimal_ensemble'][:3]}",
        f"Best overall: {hierarchy['best_overall']}",
        f"Best sCRPS: {summary['leaderboard'].index[0]}",
        f"Diverse pair: {hierarchy['best_diverse_pair']}",
        f"All models for comparison"
    ],
    'Rationale': [
        'Reduces risk through diversification',
        'Single best model by multiple criteria',
        'Maximize performance metric',
        'Test different approaches',
        'Comprehensive evaluation'
    ]
})

display(recommendations_matrix)

print("\n" + "=" * 70)
print("KEY INSIGHTS")
print("=" * 70)

# Analyze results
best_scrps = summary['leaderboard']['sCRPS_mean'].min()
worst_scrps = summary['leaderboard']['sCRPS_mean'].max()
spread = worst_scrps - best_scrps

print(f"\n1. Performance Spread:")
print(f"   Best sCRPS:  {best_scrps:.4f}")
print(f"   Worst sCRPS: {worst_scrps:.4f}")
print(f"   Spread:      {spread:.4f} ({spread/best_scrps*100:.1f}% relative)")

# Coverage analysis
well_calibrated = summary['leaderboard'][
    (summary['leaderboard']['Coverage_90'] >= 0.88) & 
    (summary['leaderboard']['Coverage_90'] <= 0.92)
]
print(f"\n2. Calibration:")
print(f"   Well-calibrated models: {len(well_calibrated)}/{len(model_names)}")
if len(well_calibrated) > 0:
    print(f"   Best calibrated: {well_calibrated.index.tolist()}")

# Ensemble benefit
best_individual = summary['leaderboard'].iloc[0]['MAE']
ensemble_mae = np.mean(np.abs(cv_results['y'] - 
                             cv_results[hierarchy['optimal_ensemble']].mean(axis=1)))
improvement = (best_individual - ensemble_mae) / best_individual * 100

print(f"\n3. Ensemble Benefit:")
print(f"   Best individual MAE: {best_individual:.6f}")
print(f"   Ensemble MAE:        {ensemble_mae:.6f}")
print(f"   Improvement:         {improvement:.1f}%")

print("\n" + "=" * 70)
print("ACTION ITEMS")
print("=" * 70)
print("\n1. Deploy ensemble of top 3 models for production")
print("2. Monitor individual model performance for degradation")
print("3. Retrain models showing calibration drift")
print("4. Consider adding more diverse architectures if correlation > 0.8")
print("5. Document selection rationale for audit trail")

## Summary

### Model Selection Framework:

1. **Single Criterion**: sCRPS for overall best
2. **Multi-Criteria**: Weight multiple metrics based on use case
3. **Constraints**: Apply hard limits for critical requirements
4. **Hierarchical**: Best overall → best by type → best ensemble
5. **Use-Case Specific**: Tailor selection to application needs

### Ensemble Strategies:

1. **Simple Average**: Robust, works well with similar models
2. **Weighted Average**: Use inverse sCRPS or optimized weights
3. **Median**: Robust to outlier predictions
4. **Optimal Size**: Test 2-5 models, diminishing returns beyond
5. **Diversity**: Low correlation (< 0.7) improves ensemble

### Best Practices:

1. Always validate ensemble out-of-sample
2. Monitor individual model performance
3. Consider computational cost vs improvement
4. Document selection criteria for reproducibility
5. Re-evaluate selection periodically

### Next Steps:

- Implement selected models in production
- Set up monitoring for model drift
- Schedule periodic re-evaluation
- See `05_performance_tuning.ipynb` for optimization